# Projet — Prédiction Ligue 1 saison 2025-2026

## Le problème
On nous demande de prédire, pour chaque match de la saison 2025-2026, le résultat parmi :
- `1` : victoire de l'équipe à domicile
- `0` : nul
- `-1` : victoire de l'équipe à l'extérieur

C'est donc un problème de **classification multiclasse à 3 classes**.

## Ce qu'on a comme données
Plusieurs fichiers CSV, principalement :
- `matchs_2013_2024.csv` — l'historique complet des matchs avec leur résultat → c'est notre jeu d'entraînement
- `match_2025.csv` — les matchs à prédire (sans résultat, c'est ce qu'on cherche)
- `clubs_fr.csv` — quelques infos sur les clubs (effectif, âge moyen…)
- `player_valuation_before_season.csv` — la valeur marchande des joueurs à différentes dates

On a aussi les compositions d'équipes (`game_lineups`) mais le sujet précise qu'on ne les a **pas** pour les matchs à prédire, donc on ne s'en sert pas dans les features — ce serait une fuite.

## Ma démarche
1. Charger les données, filtrer pour ne garder que les matchs de championnat
2. Construire des features pertinentes (ELO, forme récente, valeur du squad…)
3. Faire un split temporel pour évaluer honnêtement le modèle
4. Tester plusieurs modèles, comparer, retenir le meilleur
5. Ré-entraîner sur tout l'historique et produire le fichier de prédictions

## Point méthodo important : pas de fuite de données
Toutes les features qui dépendent du temps (ELO, forme, head-to-head) sont calculées **uniquement à partir des matchs antérieurs**. Si je trichais en regardant le futur, j'aurais un super score en validation et un modèle nul en production. C'est typiquement le piège du time series.

## Imports

Rien d'exotique : pandas/numpy pour la manipulation, scikit-learn pour les modèles.

In [ ]:
import re
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('.')
pd.set_option('display.max_columns', 50)

## 1. Chargement et nettoyage initial

Quelques opérations qu'on fait toujours en début de projet :
- on lit les CSV
- on filtre sur `competition_type == 'domestic_league'` parce qu'il y a parfois d'autres compétitions dans le fichier et on veut juste la Ligue 1
- on convertit les dates (pandas les lit en string par défaut, c'est pénible)
- on **trie par date** : c'est indispensable pour calculer l'ELO et la forme dans l'ordre chronologique

In [ ]:
matches_hist = pd.read_csv(DATA_DIR / 'matchs_2013_2024.csv')
matches_2025 = pd.read_csv(DATA_DIR / 'match_2025.csv')
clubs = pd.read_csv(DATA_DIR / 'clubs_fr.csv')
valuations = pd.read_csv(DATA_DIR / 'player_valuation_before_season.csv')

# Filtre championnat
matches_hist = matches_hist[matches_hist['competition_type'] == 'domestic_league'].copy()

# Dates au bon format
matches_hist['date'] = pd.to_datetime(matches_hist['date'])
matches_2025['date'] = pd.to_datetime(matches_2025['date'])
valuations['date'] = pd.to_datetime(valuations['date'])

# Tri chronologique
matches_hist = matches_hist.sort_values('date').reset_index(drop=True)
matches_2025 = matches_2025.sort_values('date').reset_index(drop=True)

print(f'Matchs historiques : {len(matches_hist)}')
print(f'Matchs à prédire   : {len(matches_2025)}')
matches_hist.head(3)

## 2. Regarder la distribution de la cible

Premier réflexe avant de modéliser : regarder la répartition des classes. Ça donne tout de suite la **baseline naïve** : si une classe est ultra majoritaire, prédire tout le temps cette classe donne déjà un score correct, et il faut le battre pour montrer que le modèle sert à quelque chose.

In [ ]:
print('Distribution des résultats :')
print(matches_hist['results'].value_counts(normalize=True).round(3))
# 44% de victoires à domicile -> avantage du domicile classique en foot.
# Conclusion : la baseline "toujours 1" donne déjà ~44%. À battre.

## 3. Feature engineering

C'est la partie qui demande le plus de réflexion. Le sujet dit qu'on doit choisir nous-mêmes les features utiles. Mon raisonnement :

Le résultat d'un match dépend essentiellement de :
- **le niveau global des deux équipes** → ELO, valeur marchande, stats clubs
- **leur état du moment** → forme sur les derniers matchs
- **leur histoire commune** → confrontations directes
- **des détails de contexte** → repos, lieu (domicile/extérieur)

Je vais construire des features dans chacune de ces 4 catégories.

### 3.1 ELO

C'est le système de notation utilisé aux échecs, adapté au foot. Chaque équipe a un score ; après chaque match il monte/descend selon que le résultat était attendu ou pas.

Je l'ai choisi parce que :
- ça résume tout l'historique d'une équipe en un seul chiffre
- ça s'adapte tout seul au fil du temps (pas besoin de fenêtre fixe)
- ça gère naturellement l'avantage du domicile (j'ajoute +80 à l'équipe qui reçoit dans le calcul)

**Point clé** : je sauvegarde l'ELO **avant** le match dans la feature. Si je sauvegardais après mise à jour, le modèle verrait indirectement le résultat → fuite.

In [ ]:
def compute_elo_features(matches, k=20.0, home_adv=80.0):
    # K = sensibilité de la mise à jour (20 est une valeur classique)
    # home_adv = avantage du domicile en points ELO (~80 d'après la littérature)
    elo = defaultdict(lambda: 1500.0)  # toute nouvelle équipe démarre à 1500
    home_elos, away_elos = [], []

    for row in matches.itertuples(index=False):
        h, a = row.home_club_id, row.away_club_id
        eh, ea = elo[h], elo[a]

        # On enregistre AVANT mise à jour
        home_elos.append(eh)
        away_elos.append(ea)

        # Proba attendue de victoire à domicile (formule ELO standard)
        exp_home = 1.0 / (1.0 + 10 ** (-(eh + home_adv - ea) / 400))

        # Score réel côté domicile : 1 victoire / 0.5 nul / 0 défaite
        if row.results == 1: score_home = 1.0
        elif row.results == 0: score_home = 0.5
        else: score_home = 0.0

        # Mise à jour : l'écart entre réalité et attente est réparti entre les deux équipes
        elo[h] = eh + k * (score_home - exp_home)
        elo[a] = ea + k * ((1 - score_home) - (1 - exp_home))

    return pd.DataFrame({'home_elo': home_elos, 'away_elo': away_elos}), dict(elo)

elo_df, elo_state = compute_elo_features(matches_hist)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), elo_df], axis=1)

# Pour les matchs 2025 : on prend l'ELO de chaque équipe à la fin de l'historique
matches_2025['home_elo'] = matches_2025['home_club_id'].map(lambda c: elo_state.get(c, 1500.0))
matches_2025['away_elo'] = matches_2025['away_club_id'].map(lambda c: elo_state.get(c, 1500.0))

# Vérif rapide : qui sont les meilleures équipes selon l'ELO ?
id_to_name = dict(zip(matches_hist['home_club_id'], matches_hist['home_club_name']))
id_to_name.update(dict(zip(matches_hist['away_club_id'], matches_hist['away_club_name'])))
elo_ranking = pd.Series(elo_state).sort_values(ascending=False).head(10)
print('Top 10 ELO en fin d\'historique :')
for cid, score in elo_ranking.items():
    print(f'  {id_to_name.get(cid, cid):40s} {score:.0f}')
# On devrait retrouver PSG, OM, OL, Monaco… en tête. Si ce n'est pas le cas, il y a un bug.

### 3.2 Forme récente (5 derniers matchs)

L'ELO capture le niveau "long terme", mais une équipe peut traverser une mauvaise passe ponctuelle (blessures, crise de vestiaire…). Je rajoute donc des features sur les **5 derniers matchs** de chaque équipe :
- points moyens par match
- buts marqués / encaissés / différence

Implémentation : une `deque(maxlen=5)` par équipe qui garde les 5 derniers matchs. Je lis la forme avant le match courant, puis je l'ajoute à la deque (qui éjecte automatiquement le plus ancien).

Cas limite : pour les premiers matchs d'une équipe, la deque est vide → NaN. C'est OK, le gradient boosting gère les NaN nativement, et pour la régression logistique on imputera plus tard.

In [ ]:
def compute_form_features(matches, window=5):
    history = defaultdict(lambda: deque(maxlen=window))
    rows = []

    for row in matches.itertuples(index=False):
        feats = {}
        for side, club in [('home', row.home_club_id), ('away', row.away_club_id)]:
            past = list(history[club])
            if past:
                feats[f'{side}_form_pts'] = np.mean([p['pts'] for p in past])
                feats[f'{side}_form_gf']  = np.mean([p['gf']  for p in past])
                feats[f'{side}_form_ga']  = np.mean([p['ga']  for p in past])
                feats[f'{side}_form_gd']  = feats[f'{side}_form_gf'] - feats[f'{side}_form_ga']
                feats[f'{side}_form_n']   = len(past)
            else:
                # Première apparition → on met NaN, on traitera plus tard
                for k in ('pts', 'gf', 'ga', 'gd'):
                    feats[f'{side}_form_{k}'] = np.nan
                feats[f'{side}_form_n'] = 0
        rows.append(feats)

        # Mise à jour APRÈS lecture (sinon fuite)
        if row.results == 1: pts_h, pts_a = 3, 0
        elif row.results == 0: pts_h, pts_a = 1, 1
        else: pts_h, pts_a = 0, 3
        history[row.home_club_id].append({'pts': pts_h, 'gf': row.home_club_goals, 'ga': row.away_club_goals})
        history[row.away_club_id].append({'pts': pts_a, 'gf': row.away_club_goals, 'ga': row.home_club_goals})

    return pd.DataFrame(rows), history

form_df, form_state = compute_form_features(matches_hist, window=5)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), form_df], axis=1)

# Même logique que pour l'ELO : pour 2025, on prend la forme finale
def attach_form_future(future, history, window=5):
    rows = []
    for row in future.itertuples(index=False):
        feats = {}
        for side, club in [('home', row.home_club_id), ('away', row.away_club_id)]:
            past = list(history.get(club, []))
            if past:
                feats[f'{side}_form_pts'] = np.mean([p['pts'] for p in past])
                feats[f'{side}_form_gf']  = np.mean([p['gf']  for p in past])
                feats[f'{side}_form_ga']  = np.mean([p['ga']  for p in past])
                feats[f'{side}_form_gd']  = feats[f'{side}_form_gf'] - feats[f'{side}_form_ga']
                feats[f'{side}_form_n']   = len(past)
            else:
                for k in ('pts', 'gf', 'ga', 'gd'):
                    feats[f'{side}_form_{k}'] = np.nan
                feats[f'{side}_form_n'] = 0
        rows.append(feats)
    return pd.DataFrame(rows)

matches_2025 = pd.concat([matches_2025.reset_index(drop=True), attach_form_future(matches_2025, form_state)], axis=1)
matches_hist[['date', 'home_club_name', 'away_club_name', 'home_form_pts', 'away_form_pts', 'home_elo', 'away_elo']].tail(5)

### 3.3 Head-to-head

Les confrontations directes ont souvent une dynamique propre (un club qui réussit toujours contre un autre par exemple). Je garde les **5 derniers face-à-face** entre chaque paire d'équipes, et j'en extrais :
- taux de victoire de l'équipe qui reçoit aujourd'hui
- taux de nuls

Petite astuce : j'utilise comme clé `tuple(sorted([id1, id2]))` pour que le même historique soit consulté peu importe qui est à domicile.

In [ ]:
def compute_h2h_features(matches, window=5):
    history = defaultdict(lambda: deque(maxlen=window))
    rows = []

    for row in matches.itertuples(index=False):
        key = tuple(sorted([row.home_club_id, row.away_club_id]))
        past = list(history[key])
        if past:
            home_wins = sum(1 for p in past if p['winner'] == row.home_club_id)
            draws     = sum(1 for p in past if p['winner'] == 0)
            n = len(past)
            rows.append({'h2h_home_winrate': home_wins / n,
                         'h2h_draw_rate':    draws / n,
                         'h2h_n':            n})
        else:
            rows.append({'h2h_home_winrate': np.nan, 'h2h_draw_rate': np.nan, 'h2h_n': 0})

        if   row.results ==  1: winner = row.home_club_id
        elif row.results == -1: winner = row.away_club_id
        else:                   winner = 0
        history[key].append({'winner': winner})

    return pd.DataFrame(rows), history

h2h_df, h2h_state = compute_h2h_features(matches_hist)
matches_hist = pd.concat([matches_hist.reset_index(drop=True), h2h_df], axis=1)

def attach_h2h_future(future, history):
    rows = []
    for row in future.itertuples(index=False):
        key = tuple(sorted([row.home_club_id, row.away_club_id]))
        past = list(history.get(key, []))
        if past:
            home_wins = sum(1 for p in past if p['winner'] == row.home_club_id)
            draws     = sum(1 for p in past if p['winner'] == 0)
            n = len(past)
            rows.append({'h2h_home_winrate': home_wins / n, 'h2h_draw_rate': draws / n, 'h2h_n': n})
        else:
            rows.append({'h2h_home_winrate': np.nan, 'h2h_draw_rate': np.nan, 'h2h_n': 0})
    return pd.DataFrame(rows)

matches_2025 = pd.concat([matches_2025.reset_index(drop=True), attach_h2h_future(matches_2025, h2h_state)], axis=1)

### 3.4 Valeur du squad

Hypothèse : une équipe avec des joueurs chers a tendance à mieux performer (c'est super grossier mais empiriquement ça marche). J'utilise `player_valuation_before_season.csv` et je calcule pour chaque (club, saison) :
- la **somme totale** des valeurs des joueurs
- la somme des **3 plus chers** (proxy de l'attaque vedette / des stars)
- le nombre de joueurs valorisés

Convention que j'adopte : la "saison N" couvre août N → mai N+1. Donc pour rattacher une valuation à une saison, je prends l'année calendaire de la valuation.

In [ ]:
valuations['year'] = valuations['date'].dt.year

rows = []
for (club, year), grp in valuations.groupby(['current_club_id', 'year']):
    # Pour chaque joueur, on garde sa dernière valuation de l'année (la plus récente)
    latest_per_player = grp.sort_values('date').groupby('player_id').tail(1)
    rows.append({
        'club_id': club,
        'season': year,
        'squad_value':      latest_per_player['market_value_in_eur'].sum(),
        'squad_top3_value': latest_per_player['market_value_in_eur'].nlargest(3).sum(),
        'squad_n_players':  len(latest_per_player),
    })
squad_vals = pd.DataFrame(rows)

def attach_squad_values(matches, squad_vals):
    out = matches.copy()
    # Le fichier matchs_2013_2024 a une colonne 'season', mais match_2025 ne l'a pas → on la calcule
    if 'season' not in out.columns:
        m = out['date'].dt.month
        out['season'] = np.where(m >= 7, out['date'].dt.year, out['date'].dt.year - 1)

    # Jointure côté domicile
    out = out.merge(
        squad_vals.rename(columns={'club_id': 'home_club_id',
                                    'squad_value': 'home_squad_value',
                                    'squad_top3_value': 'home_squad_top3',
                                    'squad_n_players': 'home_squad_n'}),
        on=['home_club_id', 'season'], how='left',
    )
    # Jointure côté extérieur
    out = out.merge(
        squad_vals.rename(columns={'club_id': 'away_club_id',
                                    'squad_value': 'away_squad_value',
                                    'squad_top3_value': 'away_squad_top3',
                                    'squad_n_players': 'away_squad_n'}),
        on=['away_club_id', 'season'], how='left',
    )
    return out

matches_hist = attach_squad_values(matches_hist, squad_vals)
matches_2025 = attach_squad_values(matches_2025, squad_vals)

### 3.5 Caractéristiques statiques des clubs

Le fichier `clubs_fr.csv` donne quelques infos sur l'effectif **actuel** (donc 2025) : taille, âge moyen, % d'étrangers, capacité du stade, balance des transferts. Ces variables sont surtout utiles pour les **équipes promues en L1** en 2025-2026 (Paris FC, Lorient, Metz) qui ont peu/pas d'historique récent en L1 → ELO et forme moins fiables pour elles.

Petit souci à régler : la colonne `net_transfer_record` est une chaîne du type `"€-72.30m"` ou `"+€62.11m"`. Il faut la parser en float.

In [ ]:
# Regex pour extraire signe, valeur et unité (m = millions, k = milliers)
_NET_TRANSFER_RE = re.compile(r'([+-]?)€?(-?\d+(?:\.\d+)?)([mk]?)', re.IGNORECASE)

def parse_transfer(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace(' ', '')
    m = _NET_TRANSFER_RE.search(s)
    if not m:
        return np.nan
    sign, num, unit = m.groups()
    val = float(num)
    if unit.lower() == 'm':
        val *= 1_000_000
    elif unit.lower() == 'k':
        val *= 1_000
    if sign == '-' or s.startswith('€-') or s.startswith('-'):
        val = -abs(val)
    return val

clubs['net_transfer'] = clubs['net_transfer_record'].apply(parse_transfer)
club_cols = ['club_id', 'squad_size', 'average_age', 'foreigners_percentage',
             'national_team_players', 'stadium_seats', 'net_transfer']
c = clubs[club_cols]

# On joint deux fois : une fois côté domicile, une fois côté extérieur
for side in ('home', 'away'):
    c_side = c.add_prefix(f'{side}_')
    matches_hist = matches_hist.merge(c_side, on=f'{side}_club_id', how='left')
    matches_2025 = matches_2025.merge(c_side, on=f'{side}_club_id', how='left')

matches_hist.filter(like='squad').head(3)

### 3.6 Jours de repos et features dérivées

Dernière catégorie : le contexte du match.
- **Jours de repos** : une équipe qui a joué la veille est en théorie moins fraîche qu'une autre qui a une semaine. Je calcule l'écart en jours par rapport au match précédent de chaque équipe.
- **Différences** : je rajoute `home_elo - away_elo`, `home_squad_value - away_squad_value`, etc. C'est redondant pour le gradient boosting (qui peut le déduire tout seul des deux features brutes), mais ça aide énormément la régression logistique qui est linéaire.

In [ ]:
def add_rest_days(matches):
    last_seen = {}  # dernière date connue par équipe
    rh, ra = [], []
    for row in matches.sort_values('date').itertuples(index=False):
        d = row.date
        rh.append((d - last_seen[row.home_club_id]).days if row.home_club_id in last_seen else np.nan)
        ra.append((d - last_seen[row.away_club_id]).days if row.away_club_id in last_seen else np.nan)
        last_seen[row.home_club_id] = d
        last_seen[row.away_club_id] = d
    out = matches.sort_values('date').copy()
    out['rest_days_home'] = rh
    out['rest_days_away'] = ra
    return out.sort_index()

matches_hist = add_rest_days(matches_hist)
matches_2025 = add_rest_days(matches_2025)

# Features dérivées (différences)
for df in (matches_hist, matches_2025):
    df['elo_diff']         = df['home_elo'] - df['away_elo']
    df['squad_value_diff'] = df['home_squad_value'].fillna(0) - df['away_squad_value'].fillna(0)
    df['squad_top3_diff']  = df['home_squad_top3'].fillna(0)  - df['away_squad_top3'].fillna(0)

## 4. Préparation du jeu d'entraînement

Je liste toutes les features que je vais donner au modèle. Au total ~30 variables — c'est raisonnable au vu du nombre de matchs (~4700).

Ensuite je fais un **split temporel** :
- entraînement : saisons antérieures à 2023
- validation : saisons 2023 et 2024 (les plus récentes)

**Pourquoi temporel et pas aléatoire ?** Parce que la "vraie" tâche du modèle, c'est de prédire le futur à partir du passé. Si je faisais un split aléatoire, j'aurais dans le train des matchs de la même saison que ceux du test → j'aurais déjà vu la "forme" du moment dans le train, et mon score serait artificiellement bon. C'est l'erreur classique en time series.

Je vire aussi les lignes où la forme est encore NaN (premiers matchs de chaque équipe sans historique).

In [ ]:
FEATURE_COLUMNS = [
    'elo_diff', 'home_elo', 'away_elo',
    'home_form_pts', 'away_form_pts',
    'home_form_gd', 'away_form_gd',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'h2h_home_winrate', 'h2h_draw_rate', 'h2h_n',
    'squad_value_diff', 'squad_top3_diff',
    'home_squad_value', 'away_squad_value',
    'home_squad_size', 'away_squad_size',
    'home_average_age', 'away_average_age',
    'home_foreigners_percentage', 'away_foreigners_percentage',
    'home_stadium_seats',
    'home_net_transfer', 'away_net_transfer',
    'rest_days_home', 'rest_days_away',
]

# On enlève les lignes sans cible ou sans forme calculable
df = matches_hist.dropna(subset=['results', 'home_form_pts', 'away_form_pts']).copy()

train = df[df['season'] < 2023]
val   = df[df['season'] >= 2023]

X_train, y_train = train[FEATURE_COLUMNS], train['results'].astype(int)
X_val,   y_val   = val[FEATURE_COLUMNS],   val['results'].astype(int)

print(f'Train : {len(train)} matchs (saisons {train.season.min()}-{train.season.max()})')
print(f'Val   : {len(val)} matchs (saisons {val.season.min()}-{val.season.max()})')

## 5. Entraînement et comparaison de modèles

Stratégie : je teste plusieurs modèles, et je retiens celui qui marche le mieux sur la validation. Mon plan :
1. **Baseline triviale** (toujours victoire à domicile) — pour avoir un point de référence
2. **Régression logistique** — modèle linéaire simple, bonne baseline ML
3. **Gradient boosting** — modèle non linéaire, souvent imbattable sur des données tabulaires
4. **Ensemble** des deux — moyenne des probabilités, souvent un peu meilleur que chacun pris séparément

### 5.1 Baseline triviale

In [ ]:
base_acc = (y_val == 1).mean()
print(f'[baseline] Toujours "victoire domicile" -> accuracy {base_acc:.3f}')
# C'est ce que je dois battre pour montrer que mon modèle apporte quelque chose.

### 5.2 Régression logistique

Pour les modèles linéaires il faut :
- **standardiser** les features (sinon les coefficients sont déformés par les ordres de grandeur — l'ELO est ~1500, les % entre 0 et 100, etc.)
- **imputer les NaN** (la logreg ne les gère pas) → j'utilise la médiane du train, pratique et robuste aux outliers

Je règle `C=0.3` (régularisation L2 plutôt forte) pour limiter le surapprentissage.

In [ ]:
medians = X_train.median()

logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=3000, C=0.3)),
])
logreg.fit(X_train.fillna(medians), y_train)

proba_lr = logreg.predict_proba(X_val.fillna(medians))
pred_lr  = logreg.predict(X_val.fillna(medians))
acc_lr   = accuracy_score(y_val, pred_lr)
ll_lr    = log_loss(y_val, proba_lr, labels=logreg.classes_)
print(f'[logreg] accuracy={acc_lr:.3f}  log_loss={ll_lr:.3f}')

### 5.3 Gradient Boosting (HistGradientBoostingClassifier)

Modèle basé sur des arbres de décision boostés. Avantages par rapport à la logreg :
- gère **les NaN nativement** → pas besoin d'imputer
- pas de standardisation nécessaire
- capture les interactions non linéaires

J'ai choisi les hyperparamètres pour **éviter le surapprentissage** (qui était présent dans une première version avec accuracy 49% en val mais bien plus en train) :
- `max_depth=4` : arbres pas trop profonds
- `min_samples_leaf=80` : feuilles assez grosses
- `l2_regularization=2.0` : régularisation forte
- **early stopping** activé : il arrête tout seul quand la validation interne stagne, plus besoin de tuner `max_iter` à la main

In [ ]:
gbm = HistGradientBoostingClassifier(
    max_iter=600,
    learning_rate=0.03,
    max_depth=4,
    min_samples_leaf=80,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=42,
)
gbm.fit(X_train, y_train)

proba_gbm = gbm.predict_proba(X_val)
pred_gbm  = gbm.predict(X_val)
acc_gbm   = accuracy_score(y_val, pred_gbm)
ll_gbm    = log_loss(y_val, proba_gbm, labels=gbm.classes_)
print(f'[hgbm] accuracy={acc_gbm:.3f}  log_loss={ll_gbm:.3f}  (early stop à {gbm.n_iter_} itérations)')

### 5.4 Ensemble

Astuce simple : on prend la **moyenne des probabilités** des deux modèles, puis on choisit la classe la plus probable. Comme les deux modèles font des erreurs un peu différentes (la logreg manque les interactions, le boosting peut overfitter sur des patterns rares), leur moyenne tend à être plus stable que chacun pris séparément.

In [ ]:
# Vérif : les deux modèles ont les classes dans le même ordre (-1, 0, 1) → ouf
assert list(logreg.classes_) == list(gbm.classes_)

proba_ens = 0.5 * proba_lr + 0.5 * proba_gbm
pred_ens  = gbm.classes_[proba_ens.argmax(axis=1)]
acc_ens   = accuracy_score(y_val, pred_ens)
ll_ens    = log_loss(y_val, proba_ens, labels=gbm.classes_)
print(f'[ensemble] accuracy={acc_ens:.3f}  log_loss={ll_ens:.3f}')

## 6. Tableau récap et choix du modèle final

Je sélectionne le modèle au **meilleur log-loss** plutôt qu'à la meilleure accuracy. Le log-loss pénalise les mauvaises probabilités, pas seulement les mauvaises prédictions → modèle plus robuste.

In [ ]:
results = pd.DataFrame({
    'modèle':   ['baseline', 'logreg', 'hgbm', 'ensemble'],
    'accuracy': [base_acc,   acc_lr,    acc_gbm, acc_ens],
    'log_loss': [np.nan,     ll_lr,     ll_gbm,  ll_ens],
})
print(results.round(3).to_string(index=False))
print()
print('Rapport détaillé pour l\'ensemble (le meilleur log-loss) :')
print(classification_report(y_val, pred_ens, digits=3))

**Constat à noter** : le modèle prédit quasi jamais les nuls. Le rappel sur la classe 0 est minuscule. C'est cohérent avec la littérature : les nuls sont des évènements "entre les deux" très difficiles à distinguer d'une victoire serrée. On pourrait essayer du class_weight ou de la calibration, mais ça dégraderait probablement l'accuracy globale. Je laisse comme ça et je l'assume dans la conclusion.

## 7. Importance des features

Pour comprendre **ce qui pilote** les prédictions, j'utilise la **permutation importance** sur le gradient boosting : on mélange aléatoirement chaque feature et on regarde de combien le score chute. Plus la chute est grande, plus la feature est importante.

C'est plus fiable que l'importance "native" des arbres (basée sur les gains de split) qui surévalue les features à haute cardinalité.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(gbm, X_val, y_val, n_repeats=5, random_state=42, n_jobs=1)
imp = pd.Series(r.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
print('Top 10 features les plus importantes :')
print(imp.head(10).round(4))

## 8. Prédiction finale sur 2025-2026

Maintenant que j'ai choisi le modèle (l'ensemble), je le **ré-entraîne sur tout l'historique** (train + validation). C'est important : pour la prédiction réelle, autant utiliser le maximum de données dispo. La validation servait juste à choisir le modèle et les hyperparamètres, plus à mesurer la perf.

Ensuite je prédis sur les 233 matchs de 2025-2026 et j'écris le fichier au format attendu (`game_id,results`).

In [ ]:
X_full = df[FEATURE_COLUMNS]
y_full = df['results'].astype(int)
medians_full = X_full.median()

# Logreg ré-entraînée
logreg_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=3000, C=0.3)),
])
logreg_final.fit(X_full.fillna(medians_full), y_full)

# Gradient boosting ré-entraîné (mêmes hyperparamètres que pendant la validation)
gbm_final = HistGradientBoostingClassifier(
    max_iter=600, learning_rate=0.03, max_depth=4,
    min_samples_leaf=80, l2_regularization=2.0,
    early_stopping=True, validation_fraction=0.15, n_iter_no_change=30,
    random_state=42,
)
gbm_final.fit(X_full, y_full)

# Prédictions ensemble
X_future = matches_2025[FEATURE_COLUMNS]
proba_lr_fut  = logreg_final.predict_proba(X_future.fillna(medians_full))
proba_gbm_fut = gbm_final.predict_proba(X_future)
proba_fut     = 0.5 * proba_lr_fut + 0.5 * proba_gbm_fut
preds_fut     = gbm_final.classes_[proba_fut.argmax(axis=1)]

# Écriture du fichier au format attendu (identique à sample_results.csv)
predictions = pd.DataFrame({
    'game_id': matches_2025['game_id'].values,
    'results': preds_fut.astype(int),
})
predictions.to_csv(DATA_DIR / 'predictions.csv', index=False)

# Je sauvegarde aussi les probas brutes, pratique pour analyser après coup
proba_df = pd.DataFrame(proba_fut, columns=[f'p_{c}' for c in gbm_final.classes_])
proba_df.insert(0, 'game_id', matches_2025['game_id'].values)
proba_df.to_csv(DATA_DIR / 'predictions_proba.csv', index=False)

print(f'predictions.csv écrit ({len(predictions)} lignes)')
print('Distribution des prédictions :')
print(predictions['results'].value_counts(normalize=True).round(3))

## 9. Aperçu des prédictions

Petit tableau pour visualiser quelques matchs avec les probas associées.

In [ ]:
preview = matches_2025[['date', 'home_club_name', 'away_club_name']].copy()
preview['prediction'] = preds_fut
preview['p_dom (1)']  = proba_fut[:, list(gbm_final.classes_).index(1)].round(2)
preview['p_nul (0)']  = proba_fut[:, list(gbm_final.classes_).index(0)].round(2)
preview['p_ext (-1)'] = proba_fut[:, list(gbm_final.classes_).index(-1)].round(2)
preview.head(15)

## 10. Bilan

### Ce qui a marché
- L'**ELO** et la **valeur du squad** sont les features les plus importantes — logique, ça résume bien le niveau global de chaque équipe.
- Le **gradient boosting régularisé** + l'**ensemble** avec la logreg battent les deux pris séparément en log-loss.
- Sur la validation 2023-2024, on atteint **52.5 % d'accuracy** contre 43 % pour la baseline → on apporte ~10 points, c'est honnête pour ce type de problème (la littérature plafonne autour de 50-55 % sur la prédiction de matchs de foot).

### Ce qui ne marche pas / limites
- **On ne sait pas prédire les nuls.** Le rappel sur la classe 0 est très faible. C'est un problème intrinsèque au foot : les nuls n'ont pas de signature claire dans les données. Pour mieux faire il faudrait probablement passer par un modèle séparé (genre prédire d'abord le nombre de buts puis en déduire le résultat).
- **Équipes promues en 2025-2026** : leur ELO et leur forme ne sont pas à jour vu qu'elles n'étaient pas en L1 récemment. Les variables statiques de `clubs_fr.csv` compensent partiellement.
- **Pas de compositions** pour 2025 (contrainte du sujet). Les blessures et suspensions de dernière minute ne sont pas captées.

### Pistes que j'explorerais avec plus de temps
- Approximer un **xG** (expected goals) en agrégeant les évènements de `game_events_before2025.csv` (tirs cadrés, occasions, etc.)
- **Modèle de Poisson bivarié** sur les buts marqués/encaissés, puis déduire le résultat → souvent meilleur sur les nuls
- Calibration des probabilités (Platt scaling) pour mieux refléter la distribution réelle
- **Validation croisée temporelle** (TimeSeriesSplit) au lieu d'un seul split, pour avoir une estimation plus robuste

### Livrable
Le fichier `predictions.csv` (233 lignes, format `game_id,results`) est dans le même dossier que ce notebook.